# NLP Preprocessing Engine

In [1]:
import re
from collections import Counter
from typing import Dict, List




### 1) Difference between "Love" and "love" in NLP

Without case normalization, they are treated as different tokens. This increases vocabulary size and can split frequency counts for the same semantic word. Lowercasing helps treat them consistently unless case itself is meaningful.



### 2) What happens if stopwords are not removed?

High-frequency function words (like "the", "is", "and") may dominate token counts and add noise for some models. However, stopword removal is task-dependent and should not be applied blindly.



### 3) Two real-world scenarios where removing stopwords can be harmful

1. Sentiment analysis: removing "not" can flip polarity ("not good" -> "good").

2. Search/QA systems: removing key function words can change intent in phrases and queries.



### 4) Difference between stemming and lemmatization

- Stemming: rule-based truncation to a crude root; fast but less accurate.

- Lemmatization: dictionary/morphology-based conversion to valid base form; more accurate but heavier.


In [2]:
def preprocess_text(text: str) -> Dict[str, object]:

    if text is None:

        text = ""

    if not isinstance(text, str):

        text = str(text)



    original_text = text

    text = text.lower()



    # Remove URLs and email-like patterns

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    text = re.sub(r"\b[\w.-]+@[\w.-]+\.\w+\b", " ", text)



    # Remove numbers

    text = re.sub(r"\d+", " ", text)



    # Keep letters/apostrophes/spaces only

    text = re.sub(r"[^a-z'\s]", " ", text)



    # Remove extra spaces

    text = re.sub(r"\s+", " ", text).strip()



    raw_tokens = text.split() if text else []



    def normalize_elongation(token: str) -> str:

        # Shrink elongated repeats: goooood -> good, nooooo -> noo

        token = re.sub(r"(.)\1{2,}", r"\1\1", token)

        # For short emphasis words only: soo -> so, noo -> no

        if len(token) <= 3 and len(token) >= 2 and token[-1] == token[-2]:

            token = token[:-1]

        return token



    tokens = [normalize_elongation(tok) for tok in raw_tokens]



    # Remove very short tokens <=2, except meaningful negations

    keep = {"no", "not"}

    tokens = [t for t in tokens if len(t) > 2 or t in keep]



    return {

        "original_text": original_text,

        "tokens": tokens,

        "cleaned_sentence": " ".join(tokens)

    }


In [3]:
test_sentences = [

    "Get 100% FREE access now!!!",

    "I absolutely looooved this product 😍😍",

    "Worst service ever... 0/10",

    "Call me at 9876543210",

    "This is THE best course!!!",

    "Visit https://openai.com now!",

    "Nooooo this is baaad!!!",

    "OK OK OK I got it",

    "Win $$$ now!!! Limited offer!!!",

    "I am not happy with this",

    "Broooo this app is littt 🔥🔥",

    "idk whyyy this ain't working lol"

]



processed = [preprocess_text(x) for x in test_sentences]


In [4]:
# Task 3 - Required display format
for i, row in enumerate(processed, start=1):
    print(f"Sentence {i}")
    print("Original Text:", row["original_text"])
    print("Cleaned Tokens:", row["tokens"])
    print("Cleaned Sentence:", row["cleaned_sentence"])
    print("-" * 70)

Sentence 1
Original Text: Get 100% FREE access now!!!
Cleaned Tokens: ['get', 'free', 'access', 'now']
Cleaned Sentence: get free access now
----------------------------------------------------------------------
Sentence 2
Original Text: I absolutely looooved this product 😍😍
Cleaned Tokens: ['absolutely', 'looved', 'this', 'product']
Cleaned Sentence: absolutely looved this product
----------------------------------------------------------------------
Sentence 3
Original Text: Worst service ever... 0/10
Cleaned Tokens: ['worst', 'service', 'ever']
Cleaned Sentence: worst service ever
----------------------------------------------------------------------
Sentence 4
Original Text: Call me at 9876543210
Cleaned Tokens: ['call']
Cleaned Sentence: call
----------------------------------------------------------------------
Sentence 5
Original Text: This is THE best course!!!
Cleaned Tokens: ['this', 'the', 'best', 'course']
Cleaned Sentence: this the best course
-----------------------------

In [5]:
# Task 4 - Token analytics
def token_metrics(tokens: List[str]) -> Dict[str, float]:
    total = len(tokens)
    unique = len(set(tokens))
    avg_len = round(sum(len(t) for t in tokens) / total, 2) if total else 0.0
    return {
        "total_tokens": total,
        "unique_tokens": unique,
        "average_token_length": avg_len
    }

def noise_score(raw: str, cleaned_tokens: List[str]) -> float:
    raw_count = len(re.findall(r"\S+", raw))
    if raw_count == 0:
        return 0.0
    return round((raw_count - len(cleaned_tokens)) / raw_count, 2)

analytics = []
for row in processed:
    stats = token_metrics(row["tokens"])
    stats["noise_score"] = noise_score(row["original_text"], row["tokens"])
    stats["original_text"] = row["original_text"]
    analytics.append(stats)

for a in analytics:
    print(a)

most_noise = max(analytics, key=lambda x: x["noise_score"])
most_meaningful = max(analytics, key=lambda x: x["total_tokens"])

print("\nAnalysis Answers:")
print("Which sentence had the most noise?")
print(most_noise["original_text"])
print("\nWhich sentence retained the most meaningful tokens after cleaning?")
print(most_meaningful["original_text"])

{'total_tokens': 4, 'unique_tokens': 4, 'average_token_length': 4.0, 'noise_score': 0.2, 'original_text': 'Get 100% FREE access now!!!'}
{'total_tokens': 4, 'unique_tokens': 4, 'average_token_length': 6.75, 'noise_score': 0.33, 'original_text': 'I absolutely looooved this product 😍😍'}
{'total_tokens': 3, 'unique_tokens': 3, 'average_token_length': 5.33, 'noise_score': 0.25, 'original_text': 'Worst service ever... 0/10'}
{'total_tokens': 1, 'unique_tokens': 1, 'average_token_length': 4.0, 'noise_score': 0.75, 'original_text': 'Call me at 9876543210'}
{'total_tokens': 4, 'unique_tokens': 4, 'average_token_length': 4.25, 'noise_score': 0.2, 'original_text': 'This is THE best course!!!'}
{'total_tokens': 2, 'unique_tokens': 2, 'average_token_length': 4.0, 'noise_score': 0.33, 'original_text': 'Visit https://openai.com now!'}
{'total_tokens': 3, 'unique_tokens': 3, 'average_token_length': 3.33, 'noise_score': 0.25, 'original_text': 'Nooooo this is baaad!!!'}
{'total_tokens': 1, 'unique_toke

In [6]:
# Task 5 - Frequency analysis
all_tokens = [tok for row in processed for tok in row["tokens"]]
freq = Counter(all_tokens)

print("Top 10 most frequent words:")
print(freq.most_common(10))

least_5 = sorted(freq.items(), key=lambda x: (x[1], x[0]))[:5]
print("Top 5 least frequent words:")
print(least_5)

Top 10 most frequent words:
[('this', 6), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('looved', 1), ('product', 1), ('worst', 1), ('service', 1)]
Top 5 least frequent words:
[('absolutely', 1), ('access', 1), ("ain't", 1), ('baad', 1), ('best', 1)]


In [7]:
# Task 6 - Full pipeline
def full_pipeline(text_list: List[str]) -> Dict[str, List[object]]:
    rows = [preprocess_text(t) for t in text_list]
    return {
        "tokens": [r["tokens"] for r in rows],
        "clean_sentences": [r["cleaned_sentence"] for r in rows]
    }

full_pipeline(test_sentences)

{'tokens': [['get', 'free', 'access', 'now'],
  ['absolutely', 'looved', 'this', 'product'],
  ['worst', 'service', 'ever'],
  ['call'],
  ['this', 'the', 'best', 'course'],
  ['visit', 'now'],
  ['no', 'this', 'baad'],
  ['got'],
  ['win', 'now', 'limited', 'offer'],
  ['not', 'happy', 'with', 'this'],
  ['broo', 'this', 'litt'],
  ['idk', 'whyy', 'this', "ain't", 'working', 'lol']],
 'clean_sentences': ['get free access now',
  'absolutely looved this product',
  'worst service ever',
  'call',
  'this the best course',
  'visit now',
  'no this baad',
  'got',
  'win now limited offer',
  'not happy with this',
  'broo this litt',
  "idk whyy this ain't working lol"]}

In [8]:
# Task 7 - Error handling
edge_cases = ["", "😀😂🔥", "123456789", None]

for i, e in enumerate(edge_cases, start=1):
    out = preprocess_text(e)
    print(f"Case {i}:")
    print("Input:", repr(e))
    print("Output tokens:", out["tokens"])
    print("Output sentence:", out["cleaned_sentence"])
    print("-" * 70)

Case 1:
Input: ''
Output tokens: []
Output sentence: 
----------------------------------------------------------------------
Case 2:
Input: '😀😂🔥'
Output tokens: []
Output sentence: 
----------------------------------------------------------------------
Case 3:
Input: '123456789'
Output tokens: []
Output sentence: 
----------------------------------------------------------------------
Case 4:
Input: None
Output tokens: []
Output sentence: 
----------------------------------------------------------------------
